In [128]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)

print("CONNECTED")

CONNECTED


In [129]:
import pandas as pd
query = 'select * from fighters'

data = pd.read_sql(query,conn)

C:\Users\mplan\AppData\Local\Temp\ipykernel_1092\2154704120.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query,conn)


In [130]:
data.head(1)

,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,...,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,ID
0,http://ufcstats.com/fighter-details/a683f9ddb7...,Sean Daugherty,NaN,0-2-0,0,2,0.0,"6' 0""",175 lbs.,--,...,"Dec 04, 1975",0.0,0%,0.0,0%,0.0,0%,0%,0.0,1


In [131]:
data.isnull().sum()

url            0
name           0
nickname     920
record         0
wins           0
losses         0
draws        382
Height         0
Weight         0
Reach          0
STANCE        67
DOB            0
SLpM           0
Str. Acc.      0
SApM           0
Str. Def       0
TD Avg.        0
TD Acc.        0
TD Def.        0
Sub. Avg.      0
ID             0
dtype: int64

In [132]:
data.shape

(2676, 21)

In [133]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2676 entries, 0 to 2675
Data columns (total 21 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   url        2676 non-null   str    
 1   name       2676 non-null   str    
 2   nickname   1756 non-null   str    
 3   record     2676 non-null   str    
 4   wins       2676 non-null   int64  
 5   losses     2676 non-null   int64  
 6   draws      2294 non-null   float64
 7   Height     2676 non-null   str    
 8   Weight     2676 non-null   str    
 9   Reach      2676 non-null   str    
 10  STANCE     2609 non-null   str    
 11  DOB        2676 non-null   str    
 12  SLpM       2676 non-null   str    
 13  Str. Acc.  2676 non-null   str    
 14  SApM       2676 non-null   str    
 15  Str. Def   2676 non-null   str    
 16  TD Avg.    2676 non-null   str    
 17  TD Acc.    2676 non-null   str    
 18  TD Def.    2676 non-null   str    
 19  Sub. Avg.  2676 non-null   str    
 20  ID         2676 non

In [134]:
data.drop(columns=['nickname','name','record','url'],inplace=True)

In [135]:
data.head(1)

,wins,losses,draws,Height,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,ID
0,0,2,0.0,"6' 0""",175 lbs.,--,NaN,"Dec 04, 1975",0.0,0%,0.0,0%,0.0,0%,0%,0.0,1


In [136]:
data['Height'].unique()


<StringArray>
[ '6' 0"',     '--',  '6' 5"',  '6' 3"',  '6' 2"',  '5' 8"',  '5' 7"',
 '5' 10"',  '6' 8"',  '5' 9"',  '6' 1"',  '5' 4"',  '5' 6"',  '6' 7"',
 '5' 11"',  '6' 4"',  '5' 5"',  '5' 3"',  '5' 2"', '6' 10"',  '6' 6"',
  '5' 1"',  '5' 0"', '6' 11"']
Length: 24, dtype: str

In [137]:
## πρώτα πάμε να κάνουμε το ύψος σε εκατοστά 
data['Height'] = data['Height'].replace('--', '5\' 9"')
heighs_in_cm = []
for i  in data['Height']:
    left = int(i[:2].replace("'","").strip()) * 30.48
    right = int(i[2:].replace('"','').strip()) * 2.54
    total = left + right
    heighs_in_cm.append(total)

heighs_in_cm = [round(i,2) for i in heighs_in_cm]
data['height'] = heighs_in_cm
data.drop(columns=['Height'],inplace=True)
    

In [138]:
data.head(1)

,wins,losses,draws,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,ID,height
0,0,2,0.0,175 lbs.,--,NaN,"Dec 04, 1975",0.0,0%,0.0,0%,0.0,0%,0%,0.0,1,182.88


In [139]:
## πάμε να κάνουμε και το βάρος 
data['Weight'] = data['Weight'].replace('--','175 lbs.')
kgs = []
for i in data['Weight']:
    kg = int(i.replace('lbs.','')) * 0.453592
    kgs.append(kg)

kgs = [round(i,2) for i in kgs]
data['weight_kg'] = kgs
data.drop(columns=['Weight'],inplace=True)




In [140]:
data.head(1)

,wins,losses,draws,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,ID,height,weight_kg
0,0,2,0.0,--,NaN,"Dec 04, 1975",0.0,0%,0.0,0%,0.0,0%,0%,0.0,1,182.88,79.38


In [141]:
# πάμε να κάνουμε float τα ποσοστα 

cols = ['Str. Acc.','Str. Def','TD Acc.','TD Def.']

for col in cols :

    floats = []

    for j in data[col]:
        x = int(j.replace('%',''))/100
        floats.append(x)
    
    name = col + '_total'
    data[name] = floats

        
data.drop(columns=cols,inplace=True)


In [142]:
data.tail(1)

,wins,losses,draws,Reach,STANCE,DOB,SLpM,SApM,TD Avg.,Sub. Avg.,ID,height,weight_kg,Str. Acc._total,Str. Def_total,TD Acc._total,TD Def._total
2675,12,0,0.0,"76""",Orthodox,"Jan 09, 1993",2.92,0.85,11.29,1.4,2676,180.34,70.31,0.75,0.43,0.36,1.0


In [143]:
# πάμε να φτιάξουμε το reach 
data['Reach'] = data['Reach'].replace('--','71"')
reach_cm = []
for i in data['Reach']:
    x = int(i.replace('"',''))*2.54
    reach_cm.append(x)

reach_cm = [round(i,2) for i in reach_cm]
data['reach_cm'] = reach_cm
data.drop(columns=['Reach'],inplace=True)

In [144]:
data.head(1)

,wins,losses,draws,STANCE,DOB,SLpM,SApM,TD Avg.,Sub. Avg.,ID,height,weight_kg,Str. Acc._total,Str. Def_total,TD Acc._total,TD Def._total,reach_cm
0,0,2,0.0,NaN,"Dec 04, 1975",0.0,0.0,0.0,0.0,1,182.88,79.38,0.0,0.0,0.0,0.0,180.34


In [145]:
## πάμε να κάνουμε ένα mapping  to stance 

In [146]:
data['STANCE'] = data['STANCE'].fillna('unknown')
data['STANCE'].unique()

<StringArray>
['unknown', 'Orthodox', 'Southpaw', 'Open Stance', 'Switch', 'Sideways']
Length: 6, dtype: str

In [147]:
stance_mapping = {}
for i , name in enumerate(data['STANCE'].unique()):
    if name not in stance_mapping.keys():
        stance_mapping[name] = i
stance_mapping

data['stance'] = data['STANCE'].map(stance_mapping)
data.drop(columns=['STANCE'],inplace=True)

import json 

with open(r"C:\Users\mplan\Desktop\ufc\MAPPINGS\stance_mapping.json", "w") as f:
    json.dump(stance_mapping, f, indent=4)

In [149]:
data.head(1)

,wins,losses,draws,DOB,SLpM,SApM,TD Avg.,Sub. Avg.,ID,height,weight_kg,Str. Acc._total,Str. Def_total,TD Acc._total,TD Def._total,reach_cm,stance
0,0,2,0.0,"Dec 04, 1975",0.0,0.0,0.0,0.0,1,182.88,79.38,0.0,0.0,0.0,0.0,180.34,0


In [152]:
data['DOB'].value_counts()

DOB
--              88
Jan 21, 1987     4
Sep 30, 1991     4
Apr 26, 1965     3
Sep 20, 1977     3
                ..
May 01, 1994     1
Jun 03, 1998     1
Nov 05, 1998     1
Sep 27, 2003     1
Jan 09, 1993     1
Name: count, Length: 2282, dtype: int64

In [159]:
data['DOB'] = data['DOB'].replace('--','Jan 01, 2000')
data['DOB'] = pd.to_datetime(data['DOB'])

from datetime import datetime

today = pd.Timestamp.today()

data['age'] = (today - data['DOB']).dt.days / 365.25
data['age'] = data['age'].astype(int)
data.drop(columns=['DOB'],inplace=True)


In [160]:
data.head(1)

,wins,losses,draws,SLpM,SApM,TD Avg.,Sub. Avg.,ID,height,weight_kg,Str. Acc._total,Str. Def_total,TD Acc._total,TD Def._total,reach_cm,stance,age
0,0,2,0.0,0.0,0.0,0.0,0.0,1,182.88,79.38,0.0,0.0,0.0,0.0,180.34,0,50


In [164]:
data['draws'] = data['draws'].fillna(0)

In [165]:
# ένα τελευτιαο και είμαι έτοιμος 
data['draws'] = data['draws'].astype(int)

In [166]:
data.head(1)

,wins,losses,draws,SLpM,SApM,TD Avg.,Sub. Avg.,ID,height,weight_kg,Str. Acc._total,Str. Def_total,TD Acc._total,TD Def._total,reach_cm,stance,age
0,0,2,0,0.0,0.0,0.0,0.0,1,182.88,79.38,0.0,0.0,0.0,0.0,180.34,0,50


In [167]:
data.to_csv(r'C:\Users\mplan\Desktop\ufc\TRAINING_DATA\fighters_training.csv',index=False)